In [1]:
import pandas as pd
import time
from sqlite_storage import (
    retrieve_market_data,
    calc_avg_daily_volume,
    get_top_performers,
    get_first_and_last_trade_price
)
from parquet_storage import (
    load_parquet,
    load_parquet_range,
    compute_rolling_vol,
)


## 1. SQLite3 Tasks

In [ ]:
db_path = "market_data.db"

tsla_sqlite = retrieve_market_data(
    db_path,
    symbol="TSLA",
    start_time="2025-11-17",
    end_time="2025-11-18"
)

pd.DataFrame(tsla_sqlite).head()


Retrieving market data for TSLA from 2025-11-17 to 2025-11-18...


,0,1,2,3,4,5,6,7
0,5866,2025-11-17 09:30:00,TSLA,268.31,268.51,267.95,268.07,1609
1,5867,2025-11-17 09:31:00,TSLA,268.94,269.11,268.28,269.04,4809
2,5868,2025-11-17 09:32:00,TSLA,267.70,267.94,267.69,267.92,1997
3,5869,2025-11-17 09:33:00,TSLA,268.45,268.64,268.00,268.56,3461
4,5870,2025-11-17 09:34:00,TSLA,269.01,269.57,268.21,269.23,4003


In [3]:
avg_vol_sqlite = calc_avg_daily_volume(db_path)
pd.DataFrame(avg_vol_sqlite).head()


Calculating average daily volume...


,0,1,2
0,2664.759591,AAPL,2025-11-17
1,2800.838875,AAPL,2025-11-18
2,2838.936061,AAPL,2025-11-19
3,2713.565217,AAPL,2025-11-20
4,2821.063939,AAPL,2025-11-21


In [4]:
top_sqlite = get_top_performers(
    db_path,
    n=3,
    start_time="2025-11-17",
    end_time="2025-11-21"
)
pd.DataFrame(top_sqlite, columns=["pct_return", "ticker"])


Retrieving top performers...


,pct_return,ticker
0,46.571429,GOOG
1,16.299360,AAPL
2,14.088039,MSFT


In [5]:
first_last_sqlite = get_first_and_last_trade_price(db_path)
pd.DataFrame(first_last_sqlite).head(10)


Retrieving first and last trade prices...


,0,1,2
0,AAPL,270.88,2025-11-17 09:30:00
1,AAPL,287.68,2025-11-17 16:00:00
2,AAPL,287.48,2025-11-18 09:30:00
3,AAPL,289.52,2025-11-18 16:00:00
4,AAPL,288.80,2025-11-19 09:30:00
5,AAPL,295.87,2025-11-19 16:00:00
6,AAPL,296.99,2025-11-20 09:30:00
7,AAPL,319.43,2025-11-20 16:00:00
8,AAPL,319.63,2025-11-21 09:30:00
9,AAPL,334.57,2025-11-21 16:00:00


## 2. Parquet Tasks

In [7]:
dataset = load_parquet()
dataset


In [ ]:
aapl_df = load_parquet_range(dataset, "AAPL", "2025-11-17", "2025-11-21")

# timestamp set
aapl_df = aapl_df.sort_values("timestamp")

# 5min rolling average
aapl_df["rolling_5min_close"] = aapl_df["close"].rolling(window=5).mean()

aapl_df.head(10)


,timestamp,open,high,low,close,volume,ticker,rolling_5min_close
0,2025-11-17 09:30:00,271.45,272.07,270.77,270.88,1416,AAPL,NaN
1,2025-11-17 09:31:00,269.12,269.38,269.00,269.24,3812,AAPL,NaN
2,2025-11-17 09:32:00,270.36,271.24,270.22,270.86,3046,AAPL,NaN
3,2025-11-17 09:33:00,269.47,269.61,268.77,269.28,2090,AAPL,NaN
4,2025-11-17 09:34:00,269.17,269.79,269.02,269.32,2035,AAPL,269.916
5,2025-11-17 09:35:00,270.30,271.18,270.03,270.23,3230,AAPL,269.786
6,2025-11-17 09:36:00,270.61,270.90,269.68,270.45,1676,AAPL,270.028
7,2025-11-17 09:37:00,269.73,270.36,268.80,269.52,3598,AAPL,269.760
8,2025-11-17 09:38:00,270.17,270.90,269.94,270.72,3864,AAPL,270.048
9,2025-11-17 09:39:00,270.45,271.18,269.47,270.70,3020,AAPL,270.324


In [ ]:
# set tickers
tickers = aapl_df["ticker"].unique().tolist()

rolling_vol_df = compute_rolling_vol(dataset, tickers)
rolling_vol_df.head()


,timestamp,open,high,low,close,volume,ticker,return,vol_5d
0,2025-11-17 09:30:00,271.45,272.07,270.77,270.88,1416,AAPL,NaN,NaN
1,2025-11-17 09:31:00,269.12,269.38,269.00,269.24,3812,AAPL,-0.006054,NaN
2,2025-11-17 09:32:00,270.36,271.24,270.22,270.86,3046,AAPL,0.006017,NaN
3,2025-11-17 09:33:00,269.47,269.61,268.77,269.28,2090,AAPL,-0.005833,NaN
4,2025-11-17 09:34:00,269.17,269.79,269.02,269.32,2035,AAPL,0.000149,NaN


In [ ]:
# measure SQLite
start = time.time()
_ = retrieve_market_data(db_path, "TSLA", "2025-11-17", "2025-11-18")
sqlite_time = time.time() - start

# measure Parquet
start = time.time()
_ = load_parquet_range(dataset, "TSLA", "2025-11-17", "2025-11-18")
parquet_time = time.time() - start

print(f"SQLite query time : {sqlite_time:.6f} sec")
print(f"Parquet query time: {parquet_time:.6f} sec")


Retrieving market data for TSLA from 2025-11-17 to 2025-11-18...
SQLite query time : 0.002003 sec
Parquet query time: 0.003932 sec


In [ ]:
import os

# SQLite3 db size measure 
sqlite_size = os.path.getsize("market_data.db") / (1024 * 1024)

# parquet folder size measure
parquet_size = 0
for root, dirs, files in os.walk("output_parquet"):
    for f in files:
        parquet_size += os.path.getsize(os.path.join(root, f))

parquet_size = parquet_size / (1024 * 1024)

print(f"SQLite DB Size : {sqlite_size:.2f} MB")
print(f"Parquet Size    : {parquet_size:.2f} MB")


SQLite DB Size : 0.69 MB
Parquet Size    : 0.30 MB


# Query Tasks — SQLite3 vs Parquet Results

## 3. Performance Comparison

### 3.1 Query Time (TSLA 1-Day Range)

| Engine   | Query Time |
|----------|-------------|
| SQLite   | **0.002003 sec** |
| Parquet  | **0.003932 sec** |

📝 **Observation:**  
- SQLite was slightly faster for this small-range point lookup.  
- Parquet shows overhead from loading partitions + filter pushdown.

---

### 3.2 File Size Comparison

| Storage Format | Size |
|----------------|------|
| SQLite DB File | **0.69 MB** |
| Parquet Dataset | **0.30 MB** |

📝 **Observation:**  
- Parquet is more storage-efficient due to columnar compression.  
- Even after partitioning by ticker, total size remains smaller than SQLite.

---

## 4. Summary Notes

### ✔ SQLite Strengths
- Fast point lookups (small query window)
- Simple single-file storage
- Easy to use for OLTP-like workloads

### ✔ Parquet Strengths
- Much smaller storage size (columnar compression)
- Great for analytical workloads (rolling windows, full ticker scans)
- Ideal for large datasets + scalable distributed systems (Spark, S3, etc.)

### ✔ When to Use What
- **SQLite:**  
  Backtesting DB, small-to-medium datasets, local app storage

- **Parquet:**  
  Research pipelines, machine learning features, large historical datasets, cloud-based analytics

